In [ ]:
from torchvision.datasets import VOCDetection

import torch

from sklearn.preprocessing import LabelEncoder
import torchvision.models as models
import torch.nn as nn
import torchvision.transforms as T
from sklearn.svm import LinearSVC

import selectivesearch
from torchvision import ops

import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from sklearn.svm import LinearSVC

In [ ]:
dataset     = VOCDetection(root="./data", year="2007", image_set="train", download=True)
val_dataset = VOCDetection(root="./data", year="2007", image_set="val",   download=True)
print(f"Train: {len(dataset)}  Val: {len(val_dataset)}")

In [ ]:
# classes = set()

# for _, annotation in dataset:

#     objects = annotation["annotation"]["object"]

#     for obj in objects:
#         classes.add(obj["name"])

# classes = sorted(classes)
# num_classes = len(classes) 

# print(f"{num_classes} classes: ")
# classes

In [ ]:
ALL_CLASSES = [ 'bg',
                'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
                'bus', 'car', 'cat', 'chair', 'cow',
                'diningtable', 'dog', 'horse', 'motorbike', 'person',
                'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
              ]

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
le = LabelEncoder()
le.fit(ALL_CLASSES)

alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
# rm last layer
alexnet.classifier = nn.Sequential(
    *list(alexnet.classifier.children())[:-1]
)

feature_extractor = alexnet
feature_extractor.eval()
feature_extractor.to(DEVICE)

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

In [ ]:
def parse_annotation(annotation):
    objects = annotation["annotation"]["object"]
    boxes, labels = [], []
    for obj in objects:
        b = obj["bndbox"]
        boxes.append([  int(b["xmin"]),
                        int(b["ymin"]),
                        int(b["xmax"]),
                        int(b["ymax"])
                    ])
        labels.append(obj["name"])

    return boxes, labels

In [ ]:
POS_THRESH = 0.5
NEG_THRESH = 0.3

class Proposals:
    def __init__(self, image, gt_boxes, gt_labels, transform, device="cpu"):
        self.image = image
        self.W, self.H = image.size
        self.gt_boxes = gt_boxes
        self.gt_labels = gt_labels
        self.transform = transform
        self.device = device

    def get_proposals(self, scale=300, sigma=0.9, min_size=20):
        image_np = np.array(self.image)

        _, regions = selectivesearch.selective_search(
            image_np,
            scale=scale,
            sigma=sigma,
            min_size=min_size,
        )

        proposals = []
        seen = set()

        for r in regions:
            x, y, w, h = r["rect"]

            if w < 20 or h < 20:
                continue

            if w * h > 0.9 * self.W * self.H:
                continue

            box = (x, y, x + w, y + h)

            if box not in seen:
                seen.add(box)
                proposals.append(box)

        return proposals

    def assign_labels(self, proposals):
        prop_t = torch.tensor(proposals, dtype=torch.float32)
        gt_t = torch.tensor(self.gt_boxes, dtype=torch.float32)

        iou_matrix = ops.box_iou(gt_t, prop_t)

        best_iou, best_gt = iou_matrix.max(dim=0)

        pos_idx = (best_iou >= POS_THRESH).nonzero(as_tuple=True)[0].tolist()
        neg_idx = (best_iou < NEG_THRESH).nonzero(as_tuple=True)[0].tolist()

        kept = []
        labels = []
        ious = []

        for i in pos_idx:
            kept.append(proposals[i])
            labels.append(self.gt_labels[best_gt[i].item()])
            ious.append(best_iou[i].item())

        for i in neg_idx:
            kept.append(proposals[i])
            labels.append(le.transform(['bg'])[0])
            ious.append(best_iou[i].item())

        return kept, labels, ious

    def proposals_to_tensors(self, proposals):
        tensors = []
        valid = []

        for i, (x1, y1, x2, y2) in enumerate(proposals):
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(self.W, x2)
            y2 = min(self.H, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            crop = self.image.crop((x1, y1, x2, y2))
            tensors.append(self.transform(crop))
            valid.append(i)

        if not tensors:
            return torch.empty(0, 3, 224, 224), []

        return torch.stack(tensors), valid

    def nms(self, boxes, scores, iou_thresh=0.3):
        if not boxes:
            return []

        boxes_t = torch.tensor(boxes, dtype=torch.float32)
        scores_t = torch.tensor(scores, dtype=torch.float32)

        keep = ops.nms(boxes_t, scores_t, iou_thresh).tolist()

        return [boxes[i] for i in keep]

    def visualize_proposals(
        self,
        proposals,
        max_boxes=200,
        random_sample=True,
        figsize=(10, 10),
        color="red",
        linewidth=1
    ):
        if len(proposals) == 0:
            print("No proposals.")
            return

        if random_sample and len(proposals) > max_boxes:
            proposals = random.sample(proposals, max_boxes)
        else:
            proposals = proposals[:max_boxes]

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(self.image)

        for x1, y1, x2, y2 in proposals:
            rect = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=linewidth,
                edgecolor=color,
                facecolor="none"
            )
            ax.add_patch(rect)

        ax.set_title(f"{len(proposals)} Proposals")
        ax.axis("off")
        plt.show()

    def visualize_proposals2(self, proposals, labels=None, gt_boxes=None, gt_labels=None, max_boxes=200, random_sample=True, figsize=(10, 10), linewidth=1):
        if len(proposals) == 0:
            print("No proposals.")
            return

        if random_sample and len(proposals) > max_boxes:
            indices = random.sample(range(len(proposals)), max_boxes)
            proposals = [proposals[i] for i in indices]
            labels = [labels[i] for i in indices] if labels is not None else None
        else:
            proposals = proposals[:max_boxes]
            labels = labels[:max_boxes] if labels is not None else None

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(self.image)

        if gt_boxes is not None:
            for i, (x1, y1, x2, y2) in enumerate(gt_boxes):
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='green', facecolor='none')
                ax.add_patch(rect)
                if gt_labels is not None:
                    ax.text(x1, y1, le.inverse_transform([gt_labels[i]])[0], color='green', fontsize=8, backgroundcolor='white')

        for i, (x1, y1, x2, y2) in enumerate(proposals):
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=linewidth, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            if labels is not None:
                ax.text(x1, y1, le.inverse_transform([labels[i]])[0], color='red', fontsize=8, backgroundcolor='white')

        ax.set_title(f"{len(proposals)} Proposals")
        ax.axis('off')
        plt.show()   

    def visualize_proposals3(self, proposals, labels, gt_labels, figsize=(10, 10), linewidth=2):
        if len(proposals) == 0:
            print("No proposals.")
            return

        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(self.image)

        for i, (x1, y1, x2, y2) in enumerate(proposals):
            if labels[i] in gt_labels:
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=linewidth, edgecolor='yellow', facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1, le.inverse_transform([labels[i]])[0], color='yellow', fontsize=8, backgroundcolor='black')

        ax.axis('off')
        plt.show()

In [ ]:
image, annotation = dataset[0]
gt_boxes, gt_labels = parse_annotation(annotation)
gt_labels = le.transform(gt_labels).tolist()
P = Proposals(
    image=image,
    gt_boxes=gt_boxes,
    gt_labels=gt_labels,
    transform=transform,
    device=DEVICE
)
proposals = P.get_proposals()
# P.visualize_proposals(proposals, max_boxes=200)

kept, labels, _ = P.assign_labels(proposals)
P.visualize_proposals2(kept, labels=labels, gt_boxes=gt_boxes, gt_labels=gt_labels)

P.visualize_proposals3(kept, labels=labels, gt_labels=gt_labels)
le.inverse_transform(gt_labels) , le.inverse_transform(labels)

In [ ]:
def get_training_data(image, annotation):
    gt_boxes, gt_labels = parse_annotation(annotation)
    gt_labels_enc = le.transform(gt_labels).tolist()

    P = Proposals(image=image, gt_boxes=gt_boxes, gt_labels=gt_labels_enc, transform=transform, device=DEVICE)
    proposals = P.get_proposals()
    kept_proposals, kept_labels, _ = P.assign_labels(proposals)
    tensors, valid_idx = P.proposals_to_tensors(kept_proposals)

    if tensors.shape[0] == 0: return None, None

    with torch.no_grad():
        feats = feature_extractor(tensors.to(DEVICE))

    X = feats.cpu().numpy()
    y = np.array([kept_labels[i] for i in valid_idx])

    return X, y

In [ ]:
image, annotation = dataset[0]
x, y = get_training_data(image, annotation)
x.shape , y.shape
y

In [ ]:
def prepare_dataset(dataset, max_images=None):
    X_all, y_all = [], []
    
    total = max_images if max_images is not None else len(dataset)
    
    for i in range(total):
        image, annotation = dataset[i]
        x, y = get_training_data(image, annotation)
        if x is None: continue
        
        X_all.append(x)
        y_all.append(y)
        print(f"{i+1}/{total}", end="\r")
    
    return np.vstack(X_all), np.concatenate(y_all)

In [ ]:
def train_svm(X, y):
    svm = LinearSVC(C=1.0, max_iter=10000)
    svm.fit(X, y)
    return svm

x , y = prepare_dataset(dataset ,2)
svm = train_svm(x, y)
x.shape , y.shape

In [ ]:
def predict(image, svm):
    P = Proposals(image=image, gt_boxes=[], gt_labels=[], transform=transform, device=DEVICE)
    proposals = P.get_proposals()
    tensors, valid_idx = P.proposals_to_tensors(proposals)

    if tensors.shape[0] == 0:
        return [], []

    with torch.no_grad():
        feats = feature_extractor(tensors.to(DEVICE))

    X = feats.cpu().numpy()
    scores = svm.decision_function(X)
    pred_labels = svm.predict(X)

    kept_proposals = [proposals[i] for i in valid_idx]

    bg_label = le.transform(['bg'])[0]
    fg_mask = pred_labels != bg_label

    fg_boxes  = [kept_proposals[i] for i in range(len(kept_proposals)) if fg_mask[i]]
    fg_labels = pred_labels[fg_mask]
    fg_scores = scores[fg_mask].max(axis=1)

    final_boxes = P.nms(fg_boxes, fg_scores.tolist())

    final_labels = []
    for box in final_boxes:
        idx = fg_boxes.index(box)
        final_labels.append(fg_labels[idx])

    return final_boxes, final_labels


def visualize_predictions(image, boxes, labels, gt_boxes, gt_labels):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(image)

    for (x1, y1, x2, y2), label in zip(gt_boxes, gt_labels):
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='green', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1, le.inverse_transform([label])[0], color='green', fontsize=10, backgroundcolor='white')

    if len(boxes) == 0:
        print("No predictions.")
        return
    
    for (x1, y1, x2, y2), label in zip(boxes, labels):
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1, le.inverse_transform([label])[0], color='red', fontsize=10, backgroundcolor='white')

    ax.axis('off')
    plt.show()

In [ ]:
image, annotation = val_dataset[0]
gt_boxes, gt_labels = parse_annotation(annotation)
gt_labels_enc = le.transform(gt_labels).tolist()
boxes, labels = predict(image, svm)
visualize_predictions(image, boxes, labels, gt_boxes, gt_labels_enc)